In [ ]:
# 将Human纳入循环代理
# 创建代理
import os
import sqlite3

tavily_api = os.environ.get("TAVILY_API_KEY")
tavily_api
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langchain_tavily.tavily_search import TavilySearch  # 搜索工具
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.Connection(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

tool = TavilySearch(max_results=2)
print(type(tool))
print(tool.name)



In [ ]:
from uuid import uuid4 # 全局唯一id
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage

"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""

def reduce_message(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    #
    for message in right:
        if not message.id:
            message.id = str(uuid4())

    # 将新消息个当前已经存在的消息进行合并
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # 通过id进行比较
            if existing.id == message.id:
                # i为计数器值
                merged[i] = message
                break
        else: # for...else
            merged.append(message)
    return merged
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_message]
# 配置state


In [ ]:
# 创建工具调用
tool = TavilySearch(max_results=2) # 限制最大搜索结果数

In [ ]:
from IPython.display import Image, display

class Agent:
    # 调用模型，检查操作存在，执行操作

    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        # 局部变量，仅存在于init内部
        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_openai)  # 创建llm的节点
        graph.add_node("action", self.take_action)

        # 条件边
        graph.add_conditional_edges(
            "llm",  # 传入的第一个参数
            self.exists_action,  # 决定走向
            {True: "action", False: END}  # 表示函数之间的映射方法
        )
        graph.add_edge("action", "llm")  # 添加边连接action和llm两个节点，action指向llm
        graph.set_entry_point("llm")  # 设置图的入口点
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"] # 在运行任何工具之家添加人工审批
        )

        # 传入工具列表
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)


    # 创建条件边
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0  # 判断当前是否还需要函数调用

    # 调用模型
    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []  # 存放调用工具返回的结果
        for t in tool_calls:
            # 通常情况下模型支持并行调用
            print(f"Calling Tools:{t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=result))
        print("Back to the model")
        return {"messages": results}




In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence).\
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to make \
some of the calls before asking."""

model = ChatDeepSeek(model="deepseek-v4-flash", extra_body={"thinking" : {"type": "disabled"}})
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
display(Image(abot.graph.get_graph().draw_png()))

In [ ]:
# 执行agent
messages = [HumanMessage(content="Whats the weather in Sichuan?")]
thread = {"configurable": {"thread_id": 1}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread).next

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread)

In [ ]:
abot.graph.get_state(thread).next

In [ ]:
messages = [HumanMessage("Whats the weather in Chongqing?")]
thread = {"configurable": {"thread_id": 2}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

while abot.graph.get_state(thread).next: # 当下一跳状态存在时，就会发出人工处理提示
    print(abot.graph.get_state(thread))
    _input = input("proceed?")

    if _input != "y" or _input != "Y":
        print("aborting")
        break

    # 打印后续的AIMessage
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

In [ ]:
# State Memory，使用唯一标识符标识一个状态，可以对各个状态进行访问、修改、更新
messages = [HumanMessage("Whats the weather in Beijing?")]
thread = {"configurable": {"thread_id": 3}}
for event in abot.graph.stream({"messages": messages}, thread):
    # stream将所有的操作执行出来过后再逐步打印，传入messages，对应的线程为线程3
    print(type(event))
    for v in event.values(): # 打印每一步当中的value
        print(v)

In [ ]:
# 获取当前状态
current_values = abot.graph.get_state(thread) # get_state永远获取最新的存档
current_values

In [ ]:
_id = current_values.values["messages"][-1].tool_calls[0]['id']
# 手动构造调用工具的数据包，需要获取调用工具操作的id值，然后把agrs参数进行替换，然后替换memory
current_values.values["messages"][-1].tool_calls = [
    {
        'name': 'tavily_search',
        'args': {'query': 'current weather in Tibet today',
    'search_depth': 'fast','topic': 'general'
                 },
        'id' : _id
    }
]

In [ ]:
abot.graph.update_state(thread, current_values.values) # 传入县城和新值

In [ ]:
abot.graph.get_state(thread) # 获取更新过户的状态

In [ ]:
for event in abot.graph.stream(None, thread): # None表示继续完成任务，从中断的地方继续执行
    for v in event.values():
        print(v)

In [ ]:
# 获取history
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print("--")
    states.append(state)

In [ ]:
to_replay = states[0]
to_replay

In [ ]:
to_replay.values["messages"] # 表示要恢复的目标状态

In [ ]:
for event in abot.graph.stream(None, to_replay.config):
    # 传入从该状态恢复的参数
    for k, v in event.items():
        print(v)

In [ ]:
_id = to_replay.values["messages"][-1].tool_calls[0]["id"] # 字段名称为tool_calls而非tool_call
_id

In [ ]:
to_replay.values["messages"][-1].tool_calls = [
     {
        'name': 'tavily_search',
        'args': {'query': 'current weather in Tibet, accuweather',
    'search_depth': 'fast','topic': 'general'
                 },
        'id' : _id
    }
]

In [ ]:
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

In [ ]:
for event in abot.graph.stream(None, branch_state):
    for k ,v in event.items():
        if k != "__end__":
            print(v)

In [ ]:
to_replay

In [ ]:
# 模拟响应
_id = to_replay.values['messages'][-1].tool_calls[0]["id"]

In [ ]:
state_update = {"messages" : [ToolMessage(
    tool_call_id=_id,
    name="tavily_search",
    content="54 degree celcius",
)]} # 只会追加到消息列表

In [ ]:
branch_and_add =abot.graph.update_state(
    to_replay.config,
    state_update,
    as_node="action",
)

In [ ]:
# 调用新配置的stream
for event in abot.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)